## 젝시믹스 무신사 리뷰 데이터 전처리

### 환경 설정

In [1]:
# 데이터 처리 및 분석
import pandas as pd
import ast
import numpy as np
from datetime import datetime, timedelta
import warnings
import re

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import platform

# ── 한글 폰트 설정 ──
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

# ── 출력 설정 ──
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
np.random.seed(42)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
# 데이터 불러오기
df = pd.read_csv('./check_data/xexymix_musinsa_reviews.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15086 entries, 0 to 15085
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   collect_date    15086 non-null  str    
 1   review_date     15086 non-null  str    
 2   product_id      15086 non-null  int64  
 3   product_name    15086 non-null  str    
 4   review_no       15086 non-null  int64  
 5   review_type     15086 non-null  str    
 6   rating          15086 non-null  int64  
 7   review_text     15086 non-null  str    
 8   author          15086 non-null  str    
 9   user_height     15086 non-null  int64  
 10  user_weight     15086 non-null  int64  
 11  option          15086 non-null  str    
 12  helpful         15086 non-null  int64  
 13  comment_count   15086 non-null  int64  
 14  has_image       15086 non-null  bool   
 15  image_urls      9206 non-null   str    
 16  survey_size     51 non-null     str    
 17  survey_width    6 non-null      str    
 1

In [3]:
df.head()

,collect_date,review_date,product_id,product_name,review_no,review_type,rating,review_text,author,user_height,user_weight,option,helpful,comment_count,has_image,image_urls,survey_size,survey_width,survey_comfort,survey_instep,url
0,2026-04-22 11:36:36,2026-04-09,5234703,X-스트랩 브라탑 블랙 XT7104E,83569630,일반,3,아주편하게 잘입고있어요 탑은 최고인거같아요ㅎ,놀라운신사긴팔티,164,62,L(66반~77) · 에어브라패드,0,0,False,NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5234703#revie...
1,2026-04-22 11:36:36,2026-04-09,5234702,X-스트랩 브라탑 백아이보리 XT7104E,83569620,일반,3,아주편하게 잘입고있어요 탑은 최고인거같아요ㅎ,놀라운신사긴팔티,164,62,L(66반~77) · 에어브라패드,0,0,False,NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5234702#revie...
2,2026-04-22 11:36:36,2026-03-21,5234965,X-스트랩 브라탑 그레이네이비 XT7104E,83127718,일반,5,Xl를 했어야됐나.. 꽉껴요ㅠㅠ 그래도 세탁하면 늘어나겠져..? 스브는 꽉끼게 입어...,코리락,153,53,L(66반~77) · 에어브라패드,0,0,True,https://www.musinsa.com/data/estimate/5234965_...,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5234965#revie...
3,2026-04-22 11:36:36,2026-03-15,5234965,X-스트랩 브라탑 그레이네이비 XT7104E,82977234,일반,5,색감 너무 예쁨요 ㅜㅜ 그리고 바스트도 잘 잡아주고 예뻐서 잘 입을 것 같아요,공손한보라색셋업,166,55,M(55반~66) · 에어브라패드,0,0,False,NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5234965#revie...
4,2026-04-22 11:36:36,2026-03-10,5234965,X-스트랩 브라탑 그레이네이비 XT7104E,82858241,일반,5,75c이고 m샀는데 완전 꽉끼긴해요. 그래도 입을수있긴한데 벗을때 생각하면 l사이즈...,Jerry306,158,50,M(55반~66) · 에어브라패드,0,0,True,https://www.musinsa.com/data/estimate/5234965_...,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5234965#revie...


In [4]:
# =====================================================================
# 1~4. 기본 정제 및 날짜 파생
# =====================================================================
df = df.rename(columns={
    'review_no':   'review_id',
    'review_text': 'content',
    'helpful':     'helpful_count',
    'option':      'purchase_option',
    'url':         'product_url',
})

df['platform'] = '무신사'

drop_cols = [
    'review_type', 'author', 'comment_count', 'image_urls',
    'survey_size', 'survey_width', 'survey_comfort', 'survey_instep',
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

df['collect_date'] = pd.to_datetime(df['collect_date'], errors='coerce')
df['review_date']  = pd.to_datetime(df['review_date'], errors='coerce')
df['year']  = df['review_date'].dt.year.astype('Int16')
df['month'] = df['review_date'].dt.month.astype('Int8')


# =====================================================================
# 5~7. 다중 옵션(팩 상품)의 색상 및 사이즈 동시 추출 (동적 스캔)
# =====================================================================
# 색상으로 인식하면 안 되는 금지어 사전 (기장, 신체부위, 로고 방향 등 완벽 차단)
exclude_words = {
    '쇼츠', '레깅스', '크롭탑', '브라탑', '티셔츠', '숏슬리브', '롱슬리브', '슬리브',
    '팬츠', '세트', '니삭스', '탑', '재킷', '집업', '조거팬츠', '아노락', '아우터',
    '젝시믹스', '블랙라벨', '시그니처', '패키지', 'PACK', 'SET', '에어센트', '하이디',
    '단품', '요가삭스', '삭스', '양말', '크루삭스', '하프삭스', '헤어밴드', '루프밴드', 
    '밴드', '에어브라패드', '브라패드', '패드', '기모', '롱', '숏', '스탠다드', '베이직', 
    '서포트', '볼륨', '에센셜', '우먼즈', '맨즈', '커버업', '워터레깅스', '스윔팬츠',
    '언더웨어', '심리스', '인텐션', '와이드', '스퀘어', '파인', '쿨링', '아이스페더', 
    '플랙스', '오버핏', '종', '외', '종세트', 'T', '기본',
    '양쪽', '한쪽', '왼쪽', '오른쪽', '로고', '자수', '앞면', '뒷면',
    '무릎', '발목', '종아리', '허벅지', '기장', '길이', '덮는', '아래', '위'
}

def extract_color_and_size(opt):
    if pd.isna(opt) or str(opt).strip() == '':
        return pd.NA, pd.NA
    
    parts = [p.strip() for p in str(opt).split('·')]
    colors, sizes = [], []
    
    for part in parts:
        p_clean = re.sub(r'^(?:[0-9.]*부|기모|숏|롱|스탠다드|단품|오버핏)\s*', '', part)
        
        # p_clean이 비면(예: '28부' 전체가 제거된 경우) 원본 part로 fallback
        search_str = p_clean if p_clean.strip() else part
        
        # \s* 추가로 'M (95)' 같이 공백 뒤 괄호 포맷도 처리
        # 숫자 사이즈에 부/인치/mm/cm 단위 선택적 허용 (28부, 230mm, 28인치 등)
        size_pat = r'(?i)(XS|S|M|L|XL|XXL|XXXL|2XL|3XL|S/M|M/L|F|FREE|ONE\s*SIZE|\d{2,3}(?:부|인치|mm|cm)?)(?:\s*\([^)]+\))?$'
        m = re.search(size_pat, search_str)
        
        if m:
            base = m.group(1)
            unit_m = re.search(r'(?i)(부|인치|mm|cm)$', base)
            if unit_m:
                # 단위 제거: 28부→28, 230mm→230, 28인치→28
                size_str = base[:unit_m.start()].upper().replace(' ', '')
            else:
                # 알파 사이즈: 범위 포함 전체 유지 (예: M(55반~66), M(95))
                size_str = m.group(0).upper().replace(' ', '')
            sizes.append(size_str)
            color_cand = search_str[:m.start()].strip()
        else:
            sizes.append(pd.NA)
            color_cand = p_clean
            
        if color_cand:
            m_words = re.findall(r'[가-힣]+', color_cand)
            valid_c = pd.NA
            for w in reversed(m_words):
                if w not in exclude_words:
                    valid_c = w
                    break
            colors.append(valid_c)
        else:
            colors.append(pd.NA)
    
    # 멀티컬러 조합은 가나다 정렬로 통일 (블랙,다크그레이 / 다크그레이,블랙 → 동일값)
    c_final = ', '.join(sorted([c for c in colors if pd.notna(c)]))
    s_final = ', '.join([s for s in sizes if pd.notna(s)])
    
    return c_final if c_final else pd.NA, s_final if s_final else pd.NA

extracted = df['purchase_option'].apply(extract_color_and_size)
df['purchase_option_color'] = extracted.apply(lambda x: x[0])
df['purchase_option_size']  = extracted.apply(lambda x: x[1])

# 보조 색상 추출 (옵션란에 사이즈만 있는 경우 product_name에서 색상 스캔)
def extract_fallback_color(name):
    if pd.isna(name): return pd.NA
    no_code = re.sub(r'\s+[A-Z]+\d+[A-Z0-9]*(?:_[A-Z]+\d+[A-Z0-9]*)*\s*$', '', str(name)).strip()
    m_words = re.findall(r'[가-힣]+', no_code)
    for w in reversed(m_words):
        if w not in exclude_words:
            return w
    return pd.NA

mask_na = df['purchase_option_color'].isna()
df.loc[mask_na, 'purchase_option_color'] = df.loc[mask_na, 'product_name'].apply(extract_fallback_color)


# =====================================================================
# 8. 여성 사이즈 플래그 처리 (w 접두어 치환)
# =====================================================================
# 브라탑 등 여성 전용 컵/둘레 사이즈(80~110)를 포함한 경우 타겟팅
pattern_women = r'(?i)\bW?(XS\s*\(\s*80\s*\)|S\s*\(\s*85\s*\)|M\s*\(\s*90\s*\)|L\s*\(\s*95\s*\)|XL\s*\(\s*100\s*\)|2XL\s*\(\s*105\s*\)|3XL\s*\(\s*110\s*\))'
df['women_size_yn'] = df['purchase_option_size'].str.contains(pattern_women, na=False).astype('int8')
df['purchase_option_size'] = df['purchase_option_size'].str.replace(pattern_women, r'w\1', regex=True)


# =====================================================================
# 9. 신장/체중 구간 컬럼
# =====================================================================
height_bins   = [0, 140, 145, 150, 155, 160, 165, 170, 175, 180, 185, 190, 999]
height_labels = [
    '139cm 이하', '140~144cm', '145~149cm', '150~154cm',
    '155~159cm',  '160~164cm', '165~169cm', '170~174cm',
    '175~179cm',  '180~184cm', '185~189cm', '190cm 이상',
]
weight_bins   = [0, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 999]
weight_labels = [
    '44kg 이하', '45~49kg', '50~54kg', '55~59kg', '60~64kg',
    '65~69kg',   '70~74kg', '75~79kg', '80~84kg', '85~89kg',
    '90kg 이상',
]
df['user_height_group'] = pd.cut(df['user_height'], bins=height_bins, labels=height_labels, right=False)
df['user_weight_group'] = pd.cut(df['user_weight'], bins=weight_bins, labels=weight_labels, right=False)


# =====================================================================
# 10. 데이터타입 최적화
# =====================================================================
str_cols = ['review_id', 'product_id', 'product_name', 'content', 'purchase_option', 'product_url']
cat_cols = ['platform', 'purchase_option_size', 'purchase_option_color', 'user_height_group', 'user_weight_group']

for c in str_cols: df[c] = df[c].astype('string')
for c in cat_cols: df[c] = df[c].astype('category')

df['rating']        = pd.to_numeric(df['rating'], errors='coerce').astype('Int8')
df['helpful_count'] = pd.to_numeric(df['helpful_count'], errors='coerce').astype('Int32')
df['has_image']     = pd.to_numeric(df['has_image'], errors='coerce').fillna(0).astype('int8')
df['user_height']   = pd.to_numeric(df['user_height'], errors='coerce').astype('float32')
df['user_weight']   = pd.to_numeric(df['user_weight'], errors='coerce').astype('float32')


# =====================================================================
# 11~12. 데이터 클렌징 (중복 및 숏리뷰 제거)
# =====================================================================
df = df.drop_duplicates(subset=['review_id', 'content'], keep='first')
df = df[df['content'].fillna('').str.replace(' ', '', regex=False).str.len() > 5]


# =====================================================================
# 13. 최종 컬럼 정렬
# =====================================================================
col_order = [
    'collect_date', 'platform', 'review_id', 'product_id', 'product_name',
    'review_date', 'year', 'month', 'content', 'rating', 'helpful_count',
    'has_image', 'purchase_option', 'purchase_option_size', 'purchase_option_color',
    'women_size_yn', 'user_height', 'user_weight', 'user_height_group', 'user_weight_group',
    'product_url',
]
df = df[[c for c in col_order if c in df.columns]]


# =====================================================================
# ── 결과 출력 ──
# =====================================================================
print(f"✓ 전처리 완료: {len(df):,}행 × {len(df.columns)}열")
print("\n[추출된 색상 상위 15개]")
print(df['purchase_option_color'].value_counts().head(15))
print("\n[결측치 확인]")
print(f"  color 결측치: {df['purchase_option_color'].isna().sum():,}건")
print(f"  size  결측치: {df['purchase_option_size'].isna().sum():,}건")

✓ 전처리 완료: 8,639행 × 21열

[추출된 색상 상위 15개]
purchase_option_color
블랙            3304
블랙, 블랙         521
백아이보리          198
화이트            178
백아이보리, 블랙      173
다크그레이, 블랙       91
제트차콜            84
마그넷그레이, 블랙      62
차콜              62
블랙, 오션네이비       56
블랙, 아스텔그레이      51
아이보리            48
블랙, 카본네이비       40
라이트그레이          38
블랙, 썬더그레이       37
Name: count, dtype: int64

[결측치 확인]
  color 결측치: 0건
  size  결측치: 1,289건


In [5]:
df.info()

<class 'pandas.DataFrame'>
Index: 8639 entries, 0 to 15085
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   collect_date           8639 non-null   datetime64[us]
 1   platform               8639 non-null   category      
 2   review_id              8639 non-null   string        
 3   product_id             8639 non-null   string        
 4   product_name           8639 non-null   string        
 5   review_date            8639 non-null   datetime64[us]
 6   year                   8639 non-null   Int16         
 7   month                  8639 non-null   Int8          
 8   content                8639 non-null   string        
 9   rating                 8639 non-null   Int8          
 10  helpful_count          8639 non-null   Int32         
 11  has_image              8639 non-null   int8          
 12  purchase_option        8639 non-null   string        
 13  purchase_option_si

In [6]:
# 1. 최종 결측치 확인 (content, color, size 위주)
print(df[['content', 'purchase_option_color', 'purchase_option_size']].isna().sum())

# 2. 멀티 옵션(콤마) 잘 들어갔는지 다시 한번 확인
print(df[df['purchase_option_color'].str.contains(',', na=False)][['purchase_option', 'purchase_option_color', 'purchase_option_size']].head(3))

content                     0
purchase_option_color       0
purchase_option_size     1289
dtype: int64
       purchase_option purchase_option_color purchase_option_size
267      블랙 L · 블랑실버 L              블랑실버, 블랙                 L, L
268  백아이보리 M · 슈가베이지 M          백아이보리, 슈가베이지                 M, M
269  백아이보리 M · 슈가베이지 M          백아이보리, 슈가베이지                 M, M


In [7]:
# purchase_option_size 결측치 데이터 확인
print(df['purchase_option_size'].isna().sum())
df['purchase_option_size'].value_counts(dropna=False).head(30)

1289


purchase_option_size
NaN              1289
M(55반~66)        1178
M, M              906
L                 722
S(44~55)          627
L(66반~77)         539
S, S              538
L, L              536
XL                532
M                 293
XL, XL            266
XL(77~88)         240
XXL               204
S                  99
XXL, XXL           83
S/M                80
XXXL               71
S, M               62
M, L               61
XXL(88반~99)        60
XXXL, XXXL         38
L, M               36
M, S               34
L, XL              26
XXXL(99반~100)      24
XL, L              14
XL, XXL            11
XXL, XL             9
XXL, XXXL           9
M(95)               7
Name: count, dtype: int64

In [8]:
# purchase_option_size 결측치 원인 진단
missing_mask = df['purchase_option_size'].isna()
print(f"결측치 개수: {missing_mask.sum():,}건\n")

print("[결측치 rows의 purchase_option 상위 30개]")
print(df[missing_mask]['purchase_option'].value_counts().head(30))

print("\n[결측치 rows 샘플 (purchase_option / product_name)]")
print(df[missing_mask][['purchase_option', 'product_name', 'purchase_option_size']].head(15).to_string())

결측치 개수: 1,289건

[결측치 rows의 purchase_option 상위 30개]
purchase_option
블랙 · 세트(무릎 양쪽)    322
블랙                260
화이트                41
블랙 · 블랙            34
블랙 · 단품(무릎 한쪽)     33
내추럴베이지             31
블랙 · 코코넛밀크         22
오프화이트              14
허니듀                13
백아이보리              13
블랙 · 나이트쉐도우        11
그린                 11
라인블랙               10
라인블랙 · 라인블랙         9
라인블랙 · 라인차콜         9
스킨                  9
블랙 · 문빔베이지          8
블랙 · 차콜             8
디어블루                8
브라운                 8
아이보리                7
블랙 · 차콜그레이          6
블랙 · 라인블랙           6
밀키코코넛               6
오렌지핑크               5
블랙 · 스펙트럼네이비        4
블랙 · 미네랄민트          4
코코넛밀크               4
써니라임                4
피치크림                3
Name: count, dtype: int64[pyarrow]

[결측치 rows 샘플 (purchase_option / product_name)]
    purchase_option             product_name purchase_option_size
236          내추럴베이지  볼륨 브라패드(2.5cm) XED193E1                  NaN
237          내추럴베이지  볼륨 브라패드(2.5cm) XED193E1              

#### 애초에 사이즈가 없던 제품들 확인 완료

In [9]:
# 영문 알파벳(A-Z, a-z)이 하나라도 포함된 컬러명을 모두 추출 (가장 확실한 방법)
suspicious_colors = df[
    df['purchase_option_color'].notna() &
    df['purchase_option_color'].str.contains(r'[A-Za-z]', regex=True)
][['purchase_option', 'purchase_option_color', 'purchase_option_size']].drop_duplicates()

print(f"의심 컬러 행 수: {len(suspicious_colors):,}건\n")
if len(suspicious_colors) > 0:
    print(suspicious_colors.head(20).to_string())
else:
    print("🎉 영문 사이즈 찌꺼기가 포함된 컬러가 없습니다! 완벽합니다.")

의심 컬러 행 수: 0건

🎉 영문 사이즈 찌꺼기가 포함된 컬러가 없습니다! 완벽합니다.


In [ ]:
# 데이터 저장
#df.to_csv('./final_data/xexymix_musinsa_review_final_s2024.csv', index=False)